In [ ]:
# Install packages (run once)
%pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken

In [ ]:
import os

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True) #Helpful for relocal environment locally, but not needed in production

In [ ]:
loader = TextLoader("faq.txt") # Ensure this file exists
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)
#print(os.getenv("AZURE_OPENAI_ENDPOINT"))
#print(os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"))


In [ ]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

# This creates a FAISS index object IN MEMORY
# It's stored in your computer's RAM, not on Azure or any cloud
vectorstore = FAISS.from_documents(docs, embeddings)

## Save to disk:
# vectorstore.save_local("faiss_index")

## Load from disk later:
# vectorstore = FAISS.load_local("faiss_index", embeddings)

In [ ]:
llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
# print("Chat deployment:", os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"))
# print("Embedding deployment:", os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"))
response = llm.invoke("Reply with only: success")
print(response.content)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

In [ ]:
query = "What is our return policy?"
result = qa_chain.invoke({"query": query})

In [ ]:
print("Answer:", result["result"])
print("\n--- Sources ---")

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

In [ ]:
def safe_python_repl(code: str) -> str:
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return f"Error: {str(e)}"


class PythonREPL:
    def run(self, code):
        return safe_python_repl(code)


repl = PythonREPL()

In [ ]:
prompt = PromptTemplate.from_template(
    """
    You are a smart assistant for product managers.

    Given a business-related question, return:

    1. Valid Python code using print(...) to calculate the answer.
    2. A brief explanation of the result in plain English.

    Question: {input}

    Response:
    """
)

In [ ]:
def process_pm_query(q):
    response = llm.invoke(prompt.format(input=q)).content.strip()

    if "```python" in response:
        code_block = response.split("```python")[1].split("```")[0].strip()
    else:
        code_block = next(
            (line for line in response.splitlines() if "print" in line),
            ""
        )

    try:
        result = repl.run(code_block)

        return {
            "question": q,
            "code": code_block,
            "result": result.strip(),
            "explanation": (
                response.replace(code_block, "")
                .replace("```python", "")
                .replace("```", "")
                .strip()
            ),
        }

    except Exception as e:
        return {
            "question": q,
            "code": code_block,
            "error": str(e),
            "raw_response": response,
        }

In [ ]:
questions = [
    "If we grow 8% monthly, what is our user count after 6 months starting from 10,000?",
    "We lose 25% of our 40,000 users. How many do we retain?",
    "If each user pays $10/month and we have 5,000 users, what is the monthly revenue?",
    "If revenue is $50,000 and cost is $37,000, what's our profit?",
]

In [ ]:
for q in questions:
    print(f"\nQuestion: {q}")

    output = process_pm_query(q)

    print("Code:", output.get("code", "No code"))
    print(
        "Result:",
        output.get("result", output.get("error", "Failed to execute"))
    )
    print(
        "Explanation:",
        output.get("explanation", "No explanation")
    )